### Data Modelling 

The data modelling pipeline is as follows
- Load cleaned data
- Define X & y (Independent and dependent variable)
- Split data into train/val (stratified sample)
- Convert text into numeric using TF-IDF
- Train model using Multinomial Logistic regression (baseline model)
- Evaluate model (macro F1)
- Diagnose
- Feature engineering
- Re-train
- Evaluate
- Repeat
- Final test set ((from API))

=======

CBOW vs TF-IDF


In [52]:
#import sys
#! "{sys.executable}" -m pip install nbformat

In [53]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import accuracy_score, f1_score
from sklearn import linear_model, metrics
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import wandb

In [54]:
## Data loading

clean_df = pd.read_csv("/Users/soliufatai/Documents/PersonalDocuments/Data_Science_ML_AI_Krish_Naik/Complete-Data-Science-With-Machine-Learning-And-NLP-2024-main/2-Introduction/Intro/emotion-text-ml/data/cleaned_df.csv")

In [55]:
clean_df.head()

,emotion,clean_text
0,optimism,worry is a down payment on a problem you may n...
1,anger,my roommate its okay that we cant spell becaus...
2,joy,no but thats so cute atsu was probably shy abo...
3,anger,rooneys fucking untouchable isnt he been fucki...
4,sadness,its pretty depressing when u hit pan on ur fav...


In [56]:
## Define X & y

X = clean_df['clean_text']
y = clean_df['emotion']

In [57]:
## Data splitting

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, stratify=y, random_state = 30)


In [58]:
## Convert text into numeric using tf-idf

vectorizier = TfidfVectorizer()
X_train_tfidf = vectorizier.fit_transform(X_train)
X_val_tfidf = vectorizier.transform(X_val)

In [59]:
print(X_val_tfidf)

  (np.int32(0), np.int32(535))	0.17802577919490978
  (np.int32(0), np.int32(865))	0.17729821192456163
  (np.int32(0), np.int32(907))	0.21439416999736366
  (np.int32(0), np.int32(2188))	0.5005834769345037
  (np.int32(0), np.int32(3135))	0.2841148363034326
  (np.int32(0), np.int32(3277))	0.13123632608527264
  (np.int32(0), np.int32(3644))	0.24180621759158036
  (np.int32(0), np.int32(4015))	0.30068715893669956
  (np.int32(0), np.int32(4385))	0.17876402020382703
  (np.int32(0), np.int32(4584))	0.19855209827280068
  (np.int32(0), np.int32(5149))	0.3170632362302647
  (np.int32(0), np.int32(5214))	0.29538515162006107
  (np.int32(0), np.int32(6844))	0.306614298842261
  (np.int32(0), np.int32(7069))	0.20080572229230315
  (np.int32(1), np.int32(2087))	0.7977946879768627
  (np.int32(1), np.int32(6507))	0.6029292129561314
  (np.int32(2), np.int32(199))	0.3268680271136798
  (np.int32(2), np.int32(801))	0.35112214493772576
  (np.int32(2), np.int32(876))	0.24494801900135835
  (np.int32(2), np.int32(3

In [60]:
feature_names = vectorizier.get_feature_names_out()

print(feature_names[6568])

to


In [61]:
## Convert the y_train and y_val to numeric using label encoder
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

In [62]:
## Log baseline experiment - experiement 1

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged.
    entity="fataisoliu-fs-saleeh",
    # Set the wandb project where this run will be logged.
    project="emotion-text-ml",
    name="baseline-logreg",
    # Track hyperparameters and run metadata.
    config={
        "model": "Logistic regression",
        "vectorizer": "tfidf",
        "ngram_range": (1, 1),
        "max_iter": 1000
    },
)

In [63]:
# Train baseline model

clf = linear_model.LogisticRegression(max_iter=10000, random_state = 2)
clf.fit(X_train_tfidf, y_train_enc)

y_pred = clf.predict(X_val_tfidf)

accuracy = metrics.accuracy_score(y_val_enc, y_pred) * 100
macro_f1 = metrics.f1_score(y_val_enc, y_pred, average= 'macro')

print(f"Logistic Regression model accuracy: {accuracy:.2f}%")
print(f"Logistic Regression Macro f1_score: {macro_f1}")


## Log results

wandb.log({
    "accuracy": accuracy,
    "macro_f1": macro_f1
})

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_val_enc,
        preds=y_pred,
        class_names=le.classes_
    )
})

wandb.finish()

Logistic Regression model accuracy: 66.87%
Logistic Regression Macro f1_score: 0.5760688434433853


accuracy,▁
macro_f1,▁
accuracy,66.87117
macro_f1,0.57607


In [64]:
## Match score to classes

f1_score = metrics.f1_score(y_val_enc, y_pred, average=None)

for label, score in zip(le.classes_, f1_score):
    print(f"f1_score by class {label}: {score:.4f}")

f1_score by class anger: 0.7424
f1_score by class joy: 0.5877
f1_score by class optimism: 0.3333
f1_score by class sadness: 0.6408


 The above shows f1_score by class. The f1_score for class optimism is lower compared to class anger, this is as a result of class imbalance.

Experiment two

- Add n-gram (from unigram ---> bigram)
- 

In [65]:
## Convert text into numeric using tf-idf

vectorizier_bigram = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf_bigram = vectorizier_bigram.fit_transform(X_train)
X_val_tfidf_bigram = vectorizier_bigram.transform(X_val)

In [66]:
## Log logreg bigram experiment - experiment 2

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged.
    entity="fataisoliu-fs-saleeh",
    # Set the wandb project where this run will be logged.
    project="emotion-text-ml",
    name="logreg-bigram",
    # Track hyperparameters and run metadata.
    config={
        "model": "Logistic regression",
        "vectorizer": "tfidf",
        "ngram_range": (1, 2),
        "max_iter": 1000
    },
)

In [67]:
# Train logreg bigram model

clf = linear_model.LogisticRegression(max_iter=10000, random_state = 2)
clf.fit(X_train_tfidf_bigram, y_train_enc)

y_pred_bigram = clf.predict(X_val_tfidf_bigram)

accuracy_bigram = metrics.accuracy_score(y_val_enc, y_pred_bigram) * 100
macro_f1_bigram = metrics.f1_score(y_val_enc, y_pred_bigram, average= 'macro')

print(f"Logistic Regression model accuracy: {accuracy_bigram:.2f}%")
print(f"Logistic Regression Macro f1_score: {macro_f1_bigram}")


## Log results

wandb.log({
    "accuracy": accuracy_bigram,
    "macro_f1": macro_f1_bigram
})

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_val_enc,
        preds=y_pred_bigram,
        class_names=le.classes_
    )
})

wandb.finish()

Logistic Regression model accuracy: 63.19%
Logistic Regression Macro f1_score: 0.5100387976580667


wandb: ERROR Control-C detected -- Run data was not synced


KeyboardInterrupt: 

The bigram doesn't help my model much as the f1_score dropped. There are might be several reasons for that like more noise as the bigram will introduce more features, the model might already learn better using the single word. Further investigation required

In [ ]:
## Quick sense check

len(vectorizier.vocabulary_)

7321

In [ ]:
len(vectorizier_bigram.vocabulary_)

32961

features went from 7321 to 32k. That's a feature explosion i was talking about

-- Experiment 3 - Tuning the bigram to improve perfromance

In [ ]:
## Convert text into numeric using tf-idf and tune 

vectorizier_bigram_tuned = TfidfVectorizer(ngram_range=(1, 2), min_df=2,max_df=0.95,max_features=10000)
X_train_tfidf_bigram_tuned = vectorizier_bigram_tuned.fit_transform(X_train)
X_val_tfidf_bigram_tuned = vectorizier_bigram_tuned.transform(X_val)

In [ ]:
## Log logreg bigram experiment - experiment 2

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged.
    entity="fataisoliu-fs-saleeh",
    # Set the wandb project where this run will be logged.
    project="emotion-text-ml",
    name="logreg-bigram-tuned",
    # Track hyperparameters and run metadata.
    config={
        "model": "Logistic regression",
        "vectorizer": "tfidf",
        "ngram_range": (1, 2),
        "max_iter": 1000,
        "min_df": 2,
        "max_df":0.95,
        "max_features":10000
    },
)

In [ ]:
# Train logreg bigram tuned model

clf = linear_model.LogisticRegression(max_iter=10000, random_state = 2)
clf.fit(X_train_tfidf_bigram_tuned, y_train_enc)

y_pred_bigram_tuned = clf.predict(X_val_tfidf_bigram_tuned)

accuracy_bigram_tuned = metrics.accuracy_score(y_val_enc, y_pred_bigram_tuned) * 100
macro_f1_bigram_tuned = metrics.f1_score(y_val_enc, y_pred_bigram_tuned, average= 'macro')

print(f"Logistic Regression model accuracy: {accuracy_bigram_tuned:.2f}%")
print(f"Logistic Regression Macro f1_score: {macro_f1_bigram_tuned}")


## Log results

wandb.log({
    "accuracy": accuracy_bigram_tuned,
    "macro_f1": macro_f1_bigram_tuned
})

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_val_enc,
        preds=y_pred_bigram_tuned,
        class_names=le.classes_
    )
})

wandb.finish()

Logistic Regression model accuracy: 67.02%
Logistic Regression Macro f1_score: 0.564166735539279


accuracy,▁
macro_f1,▁
accuracy,67.02454
macro_f1,0.56417


The macro_f1 score improved after tuning, however still below baseline (unigram)

-- Experiment 4 - Adding Class balancing to Logistic regression

In [ ]:
## Log logreg bigram experiment - experiment 2

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged.
    entity="fataisoliu-fs-saleeh",
    # Set the wandb project where this run will be logged.
    project="emotion-text-ml",
    name="logreg-bigram-tuned-balanced",
    # Track hyperparameters and run metadata.
    config={
        "model": "Logistic regression",
        "vectorizer": "tfidf",
        "ngram_range": (1, 2),
        "max_iter": 1000,
        "min_df": 2,
        "max_df":0.95,
        "max_features":10000,
        "class_weight":"balanced"
    },
)

In [ ]:
# Add balancing
# Becuase the dataset is imbalancing, adding balance will allow the model to pay attention to minority classes like optimism and will improve macro-f1 score

clf = linear_model.LogisticRegression(max_iter=10000, random_state = 2, class_weight='balanced')
clf.fit(X_train_tfidf_bigram_tuned, y_train_enc)

y_pred_bigram_tuned_balanced = clf.predict(X_val_tfidf_bigram_tuned)

accuracy_bigram_tuned_balanced = metrics.accuracy_score(y_val_enc, y_pred_bigram_tuned_balanced) * 100
macro_f1_bigram_tuned_balanced = metrics.f1_score(y_val_enc, y_pred_bigram_tuned_balanced, average= 'macro')

print(f"Logistic Regression model accuracy: {accuracy_bigram_tuned_balanced:.2f}%")
print(f"Logistic Regression Macro f1_score: {macro_f1_bigram_tuned_balanced}")


## Log results

wandb.log({
    "accuracy": accuracy_bigram_tuned_balanced,
    "macro_f1": macro_f1_bigram_tuned_balanced
})

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_val_enc,
        preds=y_pred_bigram_tuned_balanced,
        class_names=le.classes_
    )
})

wandb.finish()

Logistic Regression model accuracy: 67.33%
Logistic Regression Macro f1_score: 0.6156449024122967


accuracy,▁
macro_f1,▁
accuracy,67.33129
macro_f1,0.61564


Macro_f1 score has increased to 0.61, which indicated that class imbalance was affecting my base model

-- Experiment 5 - Train Linear SVM

In [ ]:
## Log logreg Linear SVM - experiment 5

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged.
    entity="fataisoliu-fs-saleeh",
    # Set the wandb project where this run will be logged.
    project="emotion-text-ml",
    name="linear-svm-balanced",
    # Track hyperparameters and run metadata.
    config={
        "model": "LinearSVC",
        "vectorizer": "tfidf",
        "kernel":"linear",
        "ngram_range": (1, 1),
        "class_weight":"balanced"
    },
)

In [ ]:
# Train Linear SVM

svm = SVC(kernel="linear", C=1, class_weight='balanced')
svm.fit(X_train_tfidf, y_train_enc)

y_pred_svm = svm.predict(X_val_tfidf)

accuracy_svm = metrics.accuracy_score(y_val_enc, y_pred_svm) * 100
macro_f1_svm = metrics.f1_score(y_val_enc, y_pred_svm, average= 'macro')

print(f"Support Vector Machine model accuracy: {accuracy_svm:.2f}%")
print(f"Support Vector Machine Macro f1_score: {macro_f1_svm}")


## Log results

wandb.log({
    "accuracy": accuracy_svm,
    "macro_f1": macro_f1_svm
})

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_val_enc,
        preds=y_pred_svm,
        class_names=le.classes_
    )
})

wandb.finish()

Support Vector Machine model accuracy: 68.10%
Support Vector Machine Macro f1_score: 0.6194858134771768


accuracy,▁
macro_f1,▁
accuracy,68.09816
macro_f1,0.61949


The svm (unigram-balanced) shows approx the same macro_f1 score as logistic regression (bigram-balanced). A higher accurancy in linear-svm-balanced

-- Experiment 6 - Use bigram and tuned tf_idf for svm

In [ ]:
## Log logreg Linear SVM bigram - experiment 6

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged.
    entity="fataisoliu-fs-saleeh",
    # Set the wandb project where this run will be logged.
    project="emotion-text-ml",
    name="linear-svm-bigram-tuned-balanced",
    # Track hyperparameters and run metadata.
    config={
        "model": "LinearSVC",
        "vectorizer": "tfidf",
        "kernel":"linear",
        "ngram_range": (1, 2),
        "class_weight":"balanced"
    },
)

In [ ]:
# Train Linear SVM

svm = SVC(kernel="linear", C=1, class_weight='balanced')
svm.fit(X_train_tfidf_bigram_tuned, y_train_enc)

y_pred_svm_bigram_tuned = svm.predict(X_val_tfidf_bigram_tuned)

accuracy_svm_bigram_tuned = metrics.accuracy_score(y_val_enc, y_pred_svm_bigram_tuned) * 100
macro_f1_svm_bigram_tuned = metrics.f1_score(y_val_enc, y_pred_svm_bigram_tuned, average= 'macro')

print(f"Support Vector Machine model accuracy: {accuracy_svm_bigram_tuned:.2f}%")
print(f"Support Vector Machine Macro f1_score: {macro_f1_svm_bigram_tuned}")


## Log results

wandb.log({
    "accuracy": accuracy_svm_bigram_tuned,
    "macro_f1": macro_f1_svm_bigram_tuned
})

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_val_enc,
        preds=y_pred_svm_bigram_tuned,
        class_names=le.classes_
    )
})

wandb.finish()

Support Vector Machine model accuracy: 67.94%
Support Vector Machine Macro f1_score: 0.6181537473739431


accuracy,▁
macro_f1,▁
accuracy,67.94479
macro_f1,0.61815


Based on the experiment, linear-svm-balanced outperforms linear-svm-bigram-balanced in terms of accurancy and macro_f1 by a very small margin.\

I have decided to go with the linear-svm balanced


## Model Evaluation

In [ ]:
## Load Test data

tweet_test_df = pd.read_csv("/Users/soliufatai/Documents/PersonalDocuments/Data_Science_ML_AI_Krish_Naik/Complete-Data-Science-With-Machine-Learning-And-NLP-2024-main/2-Introduction/Intro/emotion-text-ml/data/tweet_test.csv")

In [ ]:
tweet_test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1421 entries, 0 to 1420
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text     1421 non-null   object
 1   label    1421 non-null   int64 
 2   emotion  1421 non-null   object
dtypes: int64(1), object(2)
memory usage: 33.4+ KB


In [ ]:
tweet_test_df.head()

,text,label,emotion
0,#Deppression is real. Partners w/ #depressed p...,3,sadness
1,@user Interesting choice of words... Are you c...,0,anger
2,My visit to hospital for care triggered #traum...,3,sadness
3,@user Welcome to #MPSVT! We are delighted to h...,1,joy
4,What makes you feel #joyful?,1,joy


In [72]:
## Clean text data
import sys
sys.path.append("..")
from src.utils import clean_text
import re
from nltk.corpus import stopwords

#stopwords
stop_words = set(stopwords.words('english'))

#Keep important words
stop_words -= {"not", "no", "i"}

#clean data
tweet_test_df_copy = tweet_test_df[['text', 'emotion']].copy()

tweet_test_df_copy['clean_text'] = tweet_test_df_copy['text'].apply(clean_text)

In [73]:
tweet_test_df_copy.head()

,text,emotion,clean_text
0,#Deppression is real. Partners w/ #depressed p...,sadness,deppression is real partners w depressed peopl...
1,@user Interesting choice of words... Are you c...,anger,interesting choice of words are you confirmin...
2,My visit to hospital for care triggered #traum...,sadness,my visit to hospital for care triggered trauma...
3,@user Welcome to #MPSVT! We are delighted to h...,joy,welcome to mpsvt we are delighted to have you...
4,What makes you feel #joyful?,joy,what makes you feel joyful


In [75]:
cleaned_test_df = tweet_test_df_copy[['emotion', 'clean_text']].copy()
cleaned_test_df.head(10)

,emotion,clean_text
0,sadness,deppression is real partners w depressed peopl...
1,anger,interesting choice of words are you confirmin...
2,sadness,my visit to hospital for care triggered trauma...
3,joy,welcome to mpsvt we are delighted to have you...
4,joy,what makes you feel joyful
5,anger,i am revolting
6,sadness,rin might ever appeared gloomy but to be a mel...
7,sadness,in need of a change restless
8,sadness,cmbyn does screen august amp at miff
9,anger,get donovan out of your soccer booth hes awfu...
